# Actividad: pronóstico de alquileres de bicicletas con redes recurrentes (PyTorch)

Una empresa de bicicletas compartidas tiene que decidir cada noche cuántas bicicletas dejar en las estaciones y cuántas personas poner a trabajar al día siguiente. Para eso necesita saber **cuántas bicicletas se van a alquilar en cada una de las próximas 24 horas**.

Usamos los datos de **Capital Bikeshare** (Washington D.C.), publicados en el repositorio de la Universidad de California en Irvine (UCI): cuántas bicicletas se alquilaron en cada hora de 2011 y 2012, junto con el **clima** de esa hora (temperatura, humedad, viento, si llovía) y el **calendario** (la hora del día, si era día laboral).

**Lo nuevo frente a la Experiencia.** Allí, en cada paso de la serie entraba **un solo número** (los pasajeros del día o la temperatura). Aquí en cada hora entran **varios números**: los alquileres, el clima y el calendario. A eso se le llama una serie **multivariada**. Y como la empresa necesita 24 horas, usaremos la red de **secuencia → vector** de forma **autorregresiva**, como en la etapa 5 del notebook de arquitecturas: la red pronostica una hora, ese pronóstico se agrega a la ventana y se vuelve a usar la red. Con varias variables aparece un problema que en la Experiencia no existía: para armar la hora siguiente no basta con el pronóstico de los alquileres, también hay que decir qué clima y qué calendario tendrá esa hora.

El notebook ya trae la carga de los datos, su preparación, la clase para entrenar y las gráficas. **Nuestra labor** es completar las celdas marcadas con `TODO`, con lo que planteamos en la Reflexión:

1. **Etapa 2:** qué variables entran a la red.
2. **Etapa 4:** la red.
3. **Etapa 6:** cómo se arma la fila de la hora siguiente en el pronóstico autorregresivo.

## Etapa 1: Se carga la base de datos

In [ ]:
import copy
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
INDIGO, TEAL, CORAL = "#4f46e5", "#0d9488", "#e8590c"

Si el archivo no está, se descarga de internet y se descomprime en la carpeta `data`:

In [ ]:
data_dir = Path("data")
csv_path = data_dir / "hour.csv"
if not csv_path.exists():
    data_dir.mkdir(exist_ok=True)
    zip_path, _ = urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
        data_dir / "bike_sharing.zip")
    with zipfile.ZipFile(zip_path) as f:
        f.extractall(data_dir)

raw = pd.read_csv(csv_path)
raw.index = pd.to_datetime(raw["dteday"]) + pd.to_timedelta(raw["hr"], unit="h")   # fecha y hora de cada fila
print(raw.shape)
raw.head()

### ¿De qué trata el dataset?

Cada fila es **una hora**. Las columnas que vamos a usar:

| Columna | Qué es |
|---|---|
| `cnt` | cuántas bicicletas se alquilaron en esa hora (**lo que queremos pronosticar**) |
| `hr` | la hora del día, de 0 a 23 |
| `workingday` | 1 si es día laboral (lunes a viernes que no es festivo), 0 si es fin de semana o festivo |
| `temp`, `hum`, `windspeed` | temperatura, humedad y velocidad del viento, divididas entre su máximo (41 °C, 100 % y 67 km/h) |
| `weathersit` | el estado del tiempo: 1 despejado, 2 nublado o neblina, 3 lluvia o nieve ligeras, 4 lluvia o nieve fuertes |

Faltan algunas horas (unas 165 de 17 544, casi todas de madrugada o de días en que el sistema no funcionó). Una serie de tiempo debe tener **todas** las horas en orden, así que las agregamos: en esas horas no hubo alquileres (0) y el clima es el de la hora anterior.

Además, armamos la tabla `bikes` con las variables en unidades que se entienden (°C, %, km/h) y nombres en inglés.

In [ ]:
all_hours = pd.date_range(raw.index[0], raw.index[-1], freq="h")
print("Horas en el archivo:", len(raw), " horas entre la primera y la última:", len(all_hours))
raw = raw.reindex(all_hours)
raw["cnt"] = raw["cnt"].fillna(0)
raw = raw.ffill()   # clima y calendario de la hora anterior

bikes = pd.DataFrame({
    "rentals": raw["cnt"],                       # bicicletas alquiladas en la hora
    "temp": raw["temp"] * 41,                    # °C
    "humidity": raw["hum"] * 100,                # %
    "wind": raw["windspeed"] * 67,               # km/h
    "rain": (raw["weathersit"] >= 3).astype(float),   # 1 si llueve o nieva
    "workingday": raw["workingday"],             # 1 si es día laboral
    "hour": all_hours.hour,
}, index=all_hours)
bikes.describe().T[["mean", "std", "min", "max"]].round(1)

### ¿Cómo se ve la serie?

Arriba, los dos años completos (el promedio de cada día): hay más alquileres en verano que en invierno, y en 2012 más que en 2011, porque el sistema creció. Abajo, dos semanas de 2012, hora por hora.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6.5))
bikes["rentals"].resample("D").mean().plot(ax=ax1, color=INDIGO, lw=0.8)
ax1.set_title("Alquileres por hora (promedio de cada día), 2011 y 2012"); ax1.set_xlabel("")
weeks = bikes["2012-05-07":"2012-05-20"]
ax2.plot(weeks.index, weeks["rentals"], color=INDIGO, lw=1)
for day in pd.date_range("2012-05-07", "2012-05-20"):
    if weeks.loc[str(day.date()), "workingday"].iloc[0] == 0:   # fin de semana o festivo, en gris
        ax2.axvspan(day, day + pd.Timedelta(hours=24), color="gray", alpha=0.15)
ax2.set_title("Dos semanas de mayo de 2012 (en gris, fin de semana)")
for ax in (ax1, ax2):
    ax.set_ylabel("Bicicletas por hora"); ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

En las dos semanas se ve que **la forma del día depende del calendario**: los días laborales tienen dos picos, uno hacia las 8 de la mañana y otro hacia las 5 o 6 de la tarde (la gente va y vuelve del trabajo), y los fines de semana tienen una sola joroba al mediodía.

Las dos gráficas siguientes lo resumen con el promedio de todos los datos, y muestran también cuánto pesa el clima:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
by_hour = bikes.groupby(["workingday", "hour"])["rentals"].mean().unstack(0)
ax1.plot(by_hour.index, by_hour[1], "o-", color=INDIGO, ms=3, label="día laboral")
ax1.plot(by_hour.index, by_hour[0], "o-", color=CORAL, ms=3, label="fin de semana o festivo")
ax1.set_xlabel("Hora del día"); ax1.set_ylabel("Bicicletas por hora (promedio)")
ax1.set_xticks(range(0, 24, 3)); ax1.legend(); ax1.grid(alpha=0.4)
ax1.set_title("Según el calendario")

daytime = bikes[(bikes["hour"] >= 7) & (bikes["hour"] <= 20)]
by_rain = daytime.groupby("rain")["rentals"].mean()
ax2.bar(["sin lluvia", "con lluvia o nieve"], by_rain.values, color=[TEAL, "gray"])
ax2.set_ylabel("Bicicletas por hora (promedio, 7 a 20 h)")
ax2.set_title("Según el clima")
plt.tight_layout()
plt.show()

Con lluvia se alquilan muchas menos bicicletas. **Ni la lluvia ni el día laboral se pueden adivinar mirando solo los alquileres de las horas pasadas**: por eso vale la pena dárselos a la red como variables.

## Etapa 2: Preparación de los datos

### La hora del día, como un reloj

Si a la red le damos la hora como un número de 0 a 23, para ella las 23 y las 0 quedan muy lejos (a 23 de distancia), aunque solo las separa una hora. Para evitarlo, ponemos la hora en un **reloj**: cada hora es un punto sobre un círculo, y le damos a la red las dos coordenadas de ese punto:

$$\text{hour\_sin} = \sin\left(\frac{2\pi \cdot \text{hora}}{24}\right), \qquad \text{hour\_cos} = \cos\left(\frac{2\pi \cdot \text{hora}}{24}\right)$$

Así las 23 y las 0 quedan juntas en el círculo, igual que las 11 y las 12. En la gráfica, las 24 horas:

In [ ]:
bikes["hour_sin"] = np.sin(2 * np.pi * bikes["hour"] / 24)
bikes["hour_cos"] = np.cos(2 * np.pi * bikes["hour"] / 24)

hours = np.arange(24)
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(np.sin(2 * np.pi * hours / 24), np.cos(2 * np.pi * hours / 24), color=INDIGO)
for h in hours:
    ax.annotate(str(h), (1.13 * np.sin(2 * np.pi * h / 24), 1.13 * np.cos(2 * np.pi * h / 24)), ha="center", va="center")
ax.set_xlabel("hour_sin"); ax.set_ylabel("hour_cos"); ax.set_aspect("equal")
ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35); ax.grid(alpha=0.4)
plt.show()

### Las variables que pueden entrar a la red

| Variable | Qué es | Grupo |
|---|---|---|
| `rentals` | bicicletas alquiladas en la hora | lo que se pronostica |
| `temp` | temperatura (°C) | clima |
| `humidity` | humedad (%) | clima |
| `wind` | velocidad del viento (km/h) | clima |
| `rain` | 1 si llueve o nieva, 0 si no | clima |
| `workingday` | 1 si es día laboral, 0 si no | calendario |
| `hour_sin`, `hour_cos` | la hora del día, en el reloj | calendario |

Escojan en la celda siguiente las variables de su hipótesis. Con `features = ["rentals"]` la red ve solo los alquileres, como en la Experiencia.

In [ ]:
# TODO: escriban la lista de variables de su hipótesis (las de la tabla de arriba).
#       "rentals" debe ir de primera: es la que se pronostica.
features = ["rentals"]

### División en el tiempo y normalización

Como en la Experiencia, dividimos sin mezclar el orden:

- **Entrenamiento:** de enero de 2011 a junio de 2012 (18 meses).
- **Validación:** de julio a septiembre de 2012.
- **Test:** de octubre a diciembre de 2012.

Las variables con escalas grandes (`rentals`, `temp`, `humidity`, `wind`) se normalizan: restamos la media y dividimos entre la desviación estándar, calculadas **solo con el entrenamiento**. Las que ya están entre −1 y 1 (`rain`, `workingday`, `hour_sin`, `hour_cos`) quedan como están. `make_tensors` hace todo esto para una lista de variables (la usamos otra vez en la etapa 7).

In [ ]:
TO_SCALE = ["rentals", "temp", "humidity", "wind"]
mean = bikes[:"2012-06-30"][TO_SCALE].mean()
std = bikes[:"2012-06-30"][TO_SCALE].std()
scale = std["rentals"]   # para pasar los errores a bicicletas por hora

def make_tensors(features):
    # Devuelve los tensores de entrenamiento, validación y test con las columnas de features
    assert features[0] == "rentals", "rentals debe ser la primera variable"
    table = bikes[features].copy()
    for col in features:
        if col in TO_SCALE:
            table[col] = (table[col] - mean[col]) / std[col]
    to_tensor = lambda part: torch.tensor(part.values, dtype=torch.float32)
    return (to_tensor(table[:"2012-06-30"]), to_tensor(table["2012-07-01":"2012-09-30"]),
            to_tensor(table["2012-10-01":]))

train_data, valid_data, test_data = make_tensors(features)
n_features = len(features)
print("Horas de entrenamiento, validación y test:", len(train_data), len(valid_data), len(test_data))
print("Variables por hora:", n_features, features)

### Ventanas: quién es X y quién es y

Cada ejemplo toma las últimas **48 horas** (2 días) y la etiqueta son los alquileres de la **hora siguiente**. Es lo mismo que en la Experiencia, con una diferencia: cada $x_t$ ya no es un número sino un **vector** con las variables de la hora $t$:

- **X** son las 48 horas: $X = (x_1, \dots, x_{48})$, con $x_t = (\text{rentals}_t, \text{temp}_t, \dots)$. Su forma es `(48, n_features)`: una fila por hora y una columna por variable.
- **y** son los alquileres de la hora 49 (solo la columna `rentals`). Con `horizon=24`, y son las 24 horas siguientes; lo usamos en la etapa 6.

`BikeWindowDataset` corta la serie así. La columna 0 de los datos es `rentals`: por eso la etiqueta es `data[..., 0]`.

In [ ]:
window_length = 48

class BikeWindowDataset(Dataset):
    def __init__(self, data, window_length=48, horizon=1):
        self.data = data
        self.window_length = window_length
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.window_length - self.horizon + 1

    def __getitem__(self, idx):
        end = idx + self.window_length
        X = self.data[idx:end]                        # (48, n_features)
        y = self.data[end:end + self.horizon, 0]      # (horizon,): los alquileres de las horas siguientes
        return X, y


def stack(dataset):
    # Junta todos los ejemplos de un dataset en dos tensores (sin iterar el Dataset directamente)
    pairs = [dataset[i] for i in range(len(dataset))]
    return torch.stack([p[0] for p in pairs]), torch.stack([p[1] for p in pairs])


train_set = BikeWindowDataset(train_data)
valid_set = BikeWindowDataset(valid_data)
X, y = train_set[0]
print("Ejemplos de entrenamiento y validación:", len(train_set), len(valid_set))
print("Forma de X:", tuple(X.shape), " (horas, variables)   Forma de y:", tuple(y.shape))

La tabla muestra el primer ejemplo en las unidades originales: **cada fila es un $x_t$** (pandas oculta las filas del medio con `...`). Debajo, su etiqueta y.

In [ ]:
def to_original(X, features):
    # Pasa una ventana normalizada a las unidades originales, como tabla
    table = pd.DataFrame(X.numpy().copy(), columns=features, index=[f"x_{t + 1}" for t in range(len(X))])
    for col in features:
        if col in TO_SCALE:
            table[col] = table[col] * std[col] + mean[col]
    return table

with pd.option_context("display.max_rows", 10, "display.max_columns", 10):
    display(to_original(X, features).round(2) + 0.0)
print(f"y = alquileres en la hora siguiente a x_48: {y.item() * scale + mean['rentals']:.0f} bicicletas")

### Pronóstico de referencia

El pronóstico ingenuo: **dentro de una hora se alquilarán las mismas bicicletas que ayer a esa misma hora**. Aprovecha que el día se repite, aunque no sabe si mañana es fin de semana ni si va a llover. Es el número que nuestra red debe superar. Lo medimos con el MAE, en bicicletas por hora.

In [ ]:
def naive_forecast(X, horizon=1):
    # La hora h se pronostica con la misma hora del día anterior: (batch, horizon)
    return torch.stack([X[:, window_length - 24 + h, 0] for h in range(horizon)], dim=1)

X_valid, Y_valid = stack(valid_set)
print(f"MAE ingenuo en validación: {(naive_forecast(X_valid) - Y_valid).abs().mean() * scale:.1f} bicicletas por hora")

## Etapa 3: La clase para entrenar

Es la misma clase `Forecaster` del notebook de arquitecturas: recibe la red, y tiene `train_epoch`, `evaluate`, `fit` y `predict`. Pérdida `HuberLoss`, optimizador Adam y MAE como métrica (aquí en bicicletas por hora); se queda con los pesos de la época con menor MAE de validación.

In [ ]:
class Forecaster:
    def __init__(self, net, lr=1e-3):
        self.net = net.to(device)
        self.loss_fn = nn.HuberLoss()
        self.optimizer = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.history = {"loss": [], "mae": [], "val_loss": [], "val_mae": []}

    def train_epoch(self, loader):
        # Una época: recorre todos los minibatches y actualiza los pesos
        self.net.train()
        total_loss, total_abs, n = 0.0, 0.0, 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            y_pred = self.net(X)
            loss = self.loss_fn(y_pred, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * len(X)
            total_abs += (y_pred - y).abs().sum().item()
            n += y.numel()
        return total_loss / len(loader.dataset), total_abs / n * scale

    def evaluate(self, loader):
        # Mide el desempeño sin entrenar (sin calcular gradientes)
        self.net.eval()
        total_loss, total_abs, n = 0.0, 0.0, 0
        with torch.no_grad():
            for X, y in loader:
                X, y = X.to(device), y.to(device)
                y_pred = self.net(X)
                total_loss += self.loss_fn(y_pred, y).item() * len(X)
                total_abs += (y_pred - y).abs().sum().item()
                n += y.numel()
        return total_loss / len(loader.dataset), total_abs / n * scale

    def fit(self, train_set, valid_set, epochs=20, batch_size=64):
        train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
        valid_loader = DataLoader(valid_set, batch_size=256)
        best_mae, best_state = float("inf"), None
        pbar = tqdm(range(epochs), desc="Entrenamiento", unit="época")
        for epoch in pbar:
            loss, mae = self.train_epoch(train_loader)
            val_loss, val_mae = self.evaluate(valid_loader)
            for key, value in zip(self.history, (loss, mae, val_loss, val_mae)):
                self.history[key].append(value)
            if val_mae < best_mae:  # guardamos los mejores pesos
                best_mae, best_state = val_mae, copy.deepcopy(self.net.state_dict())
            pbar.set_postfix(mae=f"{mae:.1f}", val_mae=f"{val_mae:.1f}", mejor_val_mae=f"{best_mae:.1f}")
        self.net.load_state_dict(best_state)  # nos quedamos con los mejores pesos
        print(f"Mejor MAE de validación: {best_mae:.1f} bicicletas por hora")

    def predict(self, X):
        # Devuelve el pronóstico de la hora siguiente (normalizado) para cada ventana: (batch, 1)
        self.net.eval()
        with torch.no_grad():
            return self.net(X.to(device)).cpu()

## Etapa 4: Definición del modelo

Definan aquí la red de **secuencia → vector** que propusieron en la Reflexión. Recuerden de la Experiencia:

- La entrada tiene forma `(batch, 48, n_features)`: `input_size` es el número de variables por hora, `n_features` (en la Experiencia era 1).
- Para pronosticar un solo valor se usa la salida del **último paso** como resumen de la ventana, seguida de una capa `nn.Linear`.

In [ ]:
class BikeNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        # TODO: definan aquí las capas de su red, según la hipótesis de la Reflexión.
        # Pistas:
        #   - Una capa recurrente: nn.RNN, nn.LSTM o nn.GRU, con input_size=n_features
        #     y batch_first=True. Decidan hidden_size y num_layers.
        #   - Una capa nn.Linear al final que entregue 1 número (los alquileres de la hora siguiente).
        pass

    def forward(self, X):
        # X tiene forma (batch, 48, n_features)
        # TODO: pasen X por la capa recurrente, tomen la salida del último paso
        #       y conviértanla en el pronóstico con la capa lineal. Debe devolver (batch, 1).
        pass

In [ ]:
# Verificamos que la red entregue una salida de forma (batch, 1) y contamos sus parámetros
x, _ = next(iter(DataLoader(train_set, batch_size=64)))
model = BikeNet(n_features)
print(model)
print("Salida:", tuple(model(x).shape))
print("Parámetros:", sum(p.numel() for p in model.parameters()))

## Etapa 5: Entrenamiento y pronóstico de la hora siguiente

Entrenamos la red. Pueden cambiar la tasa de aprendizaje (`lr`) y el número de épocas.

In [ ]:
torch.manual_seed(42)
forecaster = Forecaster(BikeNet(n_features), lr=1e-3)
forecaster.fit(train_set, valid_set, epochs=20)

In [ ]:
naive_valid = ((naive_forecast(X_valid) - Y_valid).abs().mean() * scale).item()
fig, ax = plt.subplots(figsize=(8, 4))
epochs = range(1, len(forecaster.history["mae"]) + 1)
ax.plot(epochs, forecaster.history["mae"], "o-", color=INDIGO, label="entrenamiento")
ax.plot(epochs, forecaster.history["val_mae"], "o-", color=CORAL, label="validación")
ax.axhline(naive_valid, color="gray", ls=":", label="ingenuo (validación)")
ax.set_xlabel("Época"); ax.set_ylabel("MAE (bicicletas por hora)"); ax.grid(alpha=0.4); ax.legend()
plt.show()

Medimos la red en **test** (octubre a diciembre de 2012, horas que nunca vio) y la comparamos con el ingenuo:

In [ ]:
test_set = BikeWindowDataset(test_data)
X_test_1, Y_test_1 = stack(test_set)
mae_net_1 = ((forecaster.predict(X_test_1) - Y_test_1).abs().mean() * scale).item()
mae_naive_1 = ((naive_forecast(X_test_1) - Y_test_1).abs().mean() * scale).item()
print(f"MAE de test, hora siguiente, red:     {mae_net_1:.1f} bicicletas por hora")
print(f"MAE de test, hora siguiente, ingenuo: {mae_naive_1:.1f} bicicletas por hora")

## Etapa 6: Las próximas 24 horas, de forma autorregresiva

### La idea

La red da **una hora**. Para tener las 24 horas que necesita la empresa la usamos **24 veces seguidas**, como en la etapa 5 del notebook de arquitecturas: cada vez que pronostica una hora, armamos la fila de esa hora, la pegamos al final de la ventana (quitando la hora más vieja) y volvemos a usar la red.

### Lo que cambia con varias variables

En la Experiencia, la fila nueva era solo el pronóstico, porque cada paso tenía un número. Aquí cada fila tiene `n_features` números, y la red solo pronosticó **uno**: los alquileres. ¿De dónde salen los demás?

- **El calendario** (`workingday`, `hour_sin`, `hour_cos`) se conoce de antemano: sabemos qué hora será y si mañana es día laboral.
- **El clima** de la hora siguiente **no** lo conocemos. Hay que decidir qué poner: el de un pronóstico del tiempo (aquí no tenemos uno, pero podemos usar el clima que de verdad hizo, como si el pronóstico del tiempo fuera perfecto) o repetir el de la última hora conocida.

Para que puedan armar la fila, `forecast_autoregressive` le pasa a `build_next_step` la fila **real** de la hora pronosticada, `future_h`, con los alquileres borrados (`NaN`, "no es un número"): los alquileres del futuro son justo lo que no conocemos. Si por error se usan, el pronóstico da `nan` y se nota enseguida.

### El código

`forecast_autoregressive` hace el ciclo de 24 pasos para todas las ventanas de test a la vez. Ustedes completan `build_next_step`, según su hipótesis.

In [ ]:
def build_next_step(X, future_h, y_next):
    # X:        (batch, 48, n_features) la ventana actual
    # future_h: (batch, n_features) la fila REAL de la hora que se acaba de pronosticar.
    #           Su columna 0 (rentals) viene vacía (NaN): son los alquileres que no conocemos.
    # y_next:   (batch, 1) el pronóstico de la red para esa hora
    # Debe devolver (batch, n_features): la fila que se agrega al final de la ventana.
    #
    # TODO: armen la fila según lo que decidieron en la Reflexión.
    # Pistas:
    #   - La columna 0 (rentals) es el pronóstico: y_next[:, 0].
    #   - Las variables de calendario (workingday, hour_sin, hour_cos) se conocen: tómenlas de future_h.
    #   - Para el clima hay dos opciones: tomarlo de future_h (como si tuviéramos un pronóstico
    #     del tiempo perfecto) o repetir el de la última hora de la ventana, X[:, -1].
    #     La posición de una variable se obtiene con features.index("temp").
    #   - Empiecen con x_next = future_h.clone() y cambien las columnas que haga falta.
    pass

In [ ]:
horizon = 24

def forecast_autoregressive(forecaster, X, future, build_step, horizon=24):
    # X: (batch, 48, n_features); future: (batch, horizon, n_features). Devuelve (batch, horizon)
    predictions = []
    for h in range(horizon):
        y_next = forecaster.predict(X)                        # la hora siguiente: (batch, 1)
        predictions.append(y_next)
        x_next = build_step(X, future[:, h], y_next)          # la fila de esa hora: (batch, n_features)
        X = torch.cat([X[:, 1:], x_next.unsqueeze(1)], dim=1)  # corre la ventana una hora
    return torch.cat(predictions, dim=1)


def future_rows(data, n_examples):
    # Las filas reales de las 24 horas que siguen a cada ventana, sin los alquileres
    future = torch.stack([data[i + window_length:i + window_length + horizon] for i in range(n_examples)])
    future[:, :, 0] = float("nan")   # los alquileres del futuro no se conocen
    return future


test_set_24 = BikeWindowDataset(test_data, horizon=horizon)
X_test, Y_test = stack(test_set_24)          # X_test: (ejemplos, 48, n_features); Y_test: (ejemplos, 24)
future_test = future_rows(test_data, len(test_set_24))

Y_net = forecast_autoregressive(forecaster, X_test, future_test, build_next_step)
print("Forma del pronóstico:", tuple(Y_net.shape), "(ejemplos de test, 24 horas)")
assert not torch.isnan(Y_net).any(), "El pronóstico tiene NaN: build_next_step está usando los alquileres del futuro"

Veamos tres ejemplos de test. Cada ventana termina a medianoche, así que las 24 horas pronosticadas son un día completo, de 0 a 23 h. A la izquierda de la línea punteada, las últimas horas que la red leyó; a la derecha, lo que pasó y lo que pronosticó la red.

In [ ]:
DAYS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
MONTHS = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
          "septiembre", "octubre", "noviembre", "diciembre"]

def plot_day(i, ax):
    # El ejemplo i de test: las últimas 24 horas leídas, lo que pasó y el pronóstico
    start = pd.Timestamp("2012-10-01") + pd.Timedelta(hours=i + window_length)   # primera hora pronosticada
    past, future = np.arange(-23, 1), np.arange(1, horizon + 1)
    ax.plot(past, X_test[i, -24:, 0] * scale + mean["rentals"], ".-", color=INDIGO, label="lo que leyó la red")
    ax.plot(future, Y_test[i] * scale + mean["rentals"], "o-", color=TEAL, ms=4, label="lo que pasó")
    ax.plot(future, Y_net[i] * scale + mean["rentals"], "x--", color=CORAL, label="pronóstico de la red")
    ax.axvline(0.5, color="gray", ls=":")
    ax.set_title(f"{DAYS[start.day_of_week]} {start.day} de {MONTHS[start.month - 1]} de {start.year}", fontsize=10)
    ax.set_xlabel("Horas (0 = última hora conocida)"); ax.grid(alpha=0.4)

# Ventanas que terminan a medianoche: el ejemplo i pronostica el día que empieza en la hora i + 48
examples = [24 * 17, 24 * 19, 24 * 28]   # sábado 20, lunes 22 y miércoles 31 de octubre
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for i, ax in zip(examples, axes):
    plot_day(i, ax)
axes[0].set_ylabel("Bicicletas por hora")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
plt.show()

Fíjense en lo que la red **no podía saber** con solo los alquileres de las últimas horas:

- **Sábado 20:** la red leyó un viernes con los dos picos de un día laboral, y aun así pronostica la joroba del fin de semana, porque `workingday` le dice que mañana no es día laboral.
- **Lunes 22:** lo contrario: leyó un domingo y pronostica los dos picos del lunes.
- **Miércoles 31:** el 29 y el 30 de octubre el sistema estuvo cerrado por el huracán Sandy (las horas en 0 de la izquierda). La red vuelve a pronosticar un día laboral, aunque más bajo de lo que pasó.

(Si su red no usa `workingday`, estos ejemplos se verán distintos.)

## Etapa 7: ¿Valió la pena usar varias variables?

Para saberlo, comparamos el error de cada una de las 24 horas con dos puntos de referencia:

- **Ingenuo:** cada hora de mañana igual a la misma hora de hoy.
- **Solo alquileres:** la misma red (`BikeNet`, con lo que ustedes definieron), entrenada igual pero con `features = ["rentals"]`, como en la Experiencia. En el ciclo autorregresivo, la fila nueva es solo el pronóstico.

La celda siguiente entrena esa segunda red.

In [ ]:
only_rentals = ["rentals"]
train_r, valid_r, test_r = make_tensors(only_rentals)
torch.manual_seed(42)
forecaster_r = Forecaster(BikeNet(1), lr=1e-3)
forecaster_r.fit(BikeWindowDataset(train_r), BikeWindowDataset(valid_r), epochs=20)

X_test_r, _ = stack(BikeWindowDataset(test_r, horizon=horizon))
Y_only_rentals = forecast_autoregressive(forecaster_r, X_test_r, future_rows(test_r, len(X_test_r)),
                                         build_step=lambda X, future_h, y_next: y_next)

In [ ]:
forecasts = {
    f"Red con {n_features} variables": Y_net,
    "Red con solo alquileres": Y_only_rentals,
    "Ingenuo (misma hora de ayer)": naive_forecast(X_test, horizon),
}
hours_ahead = np.arange(1, horizon + 1)
fig, ax = plt.subplots(figsize=(11, 4.2))
summary = {}
for (name, Y_pred), color in zip(forecasts.items(), [CORAL, INDIGO, "gray"]):
    errors = ((Y_pred - Y_test).abs().mean(dim=0) * scale).numpy()   # MAE de cada hora del horizonte
    summary[name] = errors.mean()
    ax.plot(hours_ahead, errors, marker="o", color=color, ls="--" if name.startswith("Ingenuo") else "-", label=name)
ax.set_xlabel("Horas hacia adelante"); ax.set_ylabel("MAE de test (bicicletas por hora)")
ax.set_xticks(hours_ahead); ax.grid(alpha=0.4)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
plt.show()

pd.Series(summary, name="MAE promedio de las 24 horas (test)").round(1).to_frame()

Para leer la gráfica (los números cambian un poco en cada ejecución):

- **La hora 1** es el pronóstico de la hora siguiente de la etapa 5: ahí la red ve solo datos reales.
- **De la hora 2 en adelante**, la ventana tiene cada vez más pronósticos de la red, y el error puede acumularse, como en el notebook de arquitecturas. Comparen qué tan rápido crece cada curva.
- **El ingenuo** queda plano: su error no depende de qué tan lejos se mire, porque siempre copia el día anterior.